# Redrob Candidate Discovery & Ranking Pipeline Demo
=====================================================

This notebook demonstrates the Intelligent Candidate Discovery & Ranking Pipeline for the founding **Senior AI Engineer** role at Redrob AI.

### Core Pipeline Mechanics:
1. **Phase 1: Universal Elimination (Honeypot Filter)**: Enforces 7 structural integrity checks in `honeypot.py` (e.g. expert skill duration limits, overlapping full-time dates, impossible degree timelines, and tool release date checks) to weed out synthetic profiles prior to scoring.
2. **Phase 2: Dual-Vector Semantic Matching**: Encodes target job descriptions into positive, negative, and skill vectors, matching them semantically against duration-weighted candidate career text via local `bge-small-en-v1.5` embeddings.
3. **Phase 3: Soft YOE Scaling & Grounding Checks**: Scales outlier YOE scores softly to remain compliant with job description guidelines. Skills verification checks are performed to ensure declared skills are semantically grounded in career descriptions.
4. **Phase 4: Multi-Dimensional Behavioral Signal Modifier**: Weights candidate profiles using a clamped composite multiplier built from 20 distinct platform indicators (activity recency, response rates, assessment scores, notice periods, company type, education, relocation, and career trajectory).
5. **Phase 5: Grounded LLM Brief Generation**: Generates factual hiring recommendation briefs using a local `Qwen 2.5 1.5B GGUF` model, enforced by technology-grounding post-processing filters to prevent LLM hallucinations.

### Google Colab Environment Setup (Automatic)
If running in Google Colab, we clone the Git repository, navigate into the directory, and install the required dependencies automatically. If running locally, this step is skipped.

In [1]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Setting up workspace...")
    # 1. Clone the repository
    !git clone https://github.com/Karan825/RetrievAl.git

    # 2. Change directory into the repository
    %cd RetrievAl

    # 3. Install llama-cpp-python from precompiled CPU wheels (avoids 10+ min build time)
    !pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

    # 4. Install remaining required packages
    !pip install -r requirements.txt
else:
    print("Running in local workspace. Skipping Colab setup.")

Running in Google Colab. Setting up workspace...
Cloning into 'RetrievAl'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 202 (delta 109), reused 150 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 994.73 KiB | 16.31 MiB/s, done.
Resolving deltas: 100% (109/109), done.
/content/RetrievAl
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.6/22.6 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.4 MB/s eta 0:00:00


### Step 1: System Check
We verify the existence of `sample_candidates.json` and the pre-computed JD embeddings/metadata files.

In [2]:
import os
import sys
from pathlib import Path

# Paths to verify
sample_candidates = Path("sample_candidates.json")
jd_embeddings = Path("vrd/jd_embeddings.npz")
jd_metadata = Path("vrd/jd_metadata.json")

print("Checking required files:")
print(f"  sample_candidates.json : {'EXISTS' if sample_candidates.exists() else 'MISSING'}")
print(f"  vrd/jd_embeddings.npz  : {'EXISTS' if jd_embeddings.exists() else 'MISSING'}")
print(f"  vrd/jd_metadata.json   : {'EXISTS' if jd_metadata.exists() else 'MISSING'}")

if not (sample_candidates.exists() and jd_embeddings.exists() and jd_metadata.exists()):
    raise FileNotFoundError("Required files are missing! Please check the workspace setup.")

Checking required files:
  sample_candidates.json : EXISTS
  vrd/jd_embeddings.npz  : EXISTS
  vrd/jd_metadata.json   : EXISTS


### Step 2: Convert JSON Array to JSONL Format
The main ranking pipeline expects candidate data in the JSONL format (one candidate JSON object per line). We convert `sample_candidates.json` into a temporary `temp_candidates.jsonl` file.

In [3]:
import json

print("Converting sample_candidates.json to JSONL format...")
with open("sample_candidates.json", "r", encoding="utf-8") as f:
    candidates = json.load(f)

with open("temp_candidates.jsonl", "w", encoding="utf-8") as f:
    for c in candidates:
        f.write(json.dumps(c) + "\n")

print(f"Success! Converted {len(candidates)} candidates and saved to temp_candidates.jsonl.")

Converting sample_candidates.json to JSONL format...
Success! Converted 50 candidates and saved to temp_candidates.jsonl.


### Step 3: Run the Ranking Pipeline
We run the ranker CLI (`rank.py`) with `temp_candidates.jsonl` as input and output the results to `submission.csv`.

In [4]:
# Execute the ranking wrapper script
!python rank.py --candidates temp_candidates.jsonl --out submission.csv

[rank.py] Launching main ranker...
[main_ranker] Loading JD Brain from /content/RetrievAl/vrd/jd_embeddings.npz...
Fetching 14 files: 100% 14/14 [00:03<00:00,  3.72it/s]
Download complete: 100% 401M/401M [00:03<00:00, 168MB/s]                BGE model download complete!

qwen2.5-1.5b-instruct-q4_k_m.gguf:   0% 0.00/1.12G [00:00<?, ?B/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:   1% 5.91M/1.12G [00:01<05:33, 3.33MB/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:  12% 135M/1.12G [00:02<00:10, 90.0MB/s] 
qwen2.5-1.5b-instruct-q4_k_m.gguf:  25% 277M/1.12G [00:03<00:09, 92.2MB/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:  36% 403M/1.12G [00:04<00:05, 128MB/s] 
qwen2.5-1.5b-instruct-q4_k_m.gguf:  69% 768M/1.12G [00:06<00:02, 131MB/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:  82% 913M/1.12G [00:07<00:01, 154MB/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf: 100% 1.12G/1.12G [00:08<00:00, 140MB/s]
Qwen model download complete!
[main_ranker] Loading Local Embedding Model (BGE)...

Loading weights: 100% 199/199 [00:00<00:00, 5751.88i

### Step 4: Display the Ranked Output
We read the generated `submission.csv` and display the top ranked candidates in a clean Pandas DataFrame.

In [5]:
import pandas as pd

# Load the generated submission file
df = pd.read_csv("submission.csv")

# Set display options for clean reading
pd.set_option("display.max_colwidth", None)

print(f"Total Ranked Candidates: {len(df)}")
df.head(10)

Total Ranked Candidates: 3


,candidate_id,rank,score,reasoning
0,CAND_0000031,1,1.0000,"Ranked #1 for the Senior AI Engineer at Redrob AI, Ela Singh stands out as they are currently working as a Recommendation Systems Engineer at Swiggy (including relevant experience at Swiggy, Mad Street Den), possessing 6.0 YOE (matching the target 5-9 bracket), demonstrating capabilities in Pinecone, Embeddings, Information Retrieval, and showing strong recruiter responsiveness (91% response rate). They represent a strong, highly aligned technical fit for our founding team."
1,CAND_0000032,2,0.1874,"For the Senior AI Engineer position at Redrob AI, Pranav Agarwal is placed at #2 because they are currently working as a .NET Developer at Cognizant, possessing 8.1 YOE (matching the target 5-9 bracket), demonstrating capabilities in Embeddings, Python, OpenCV, and with moderate recruiter responsiveness (69% response rate). They represent a strong, highly aligned technical fit for our founding team."
2,CAND_0000038,3,0.0000,"Ranked #3 for the Senior AI Engineer at Redrob AI, Myra Trivedi stands out as they are currently working as a Java Developer at Swiggy, possessing 6.7 YOE (matching the target 5-9 bracket), demonstrating capabilities in Weaviate, GraphQL, Azure, and with lower recruiter responsiveness (20% response rate — may need proactive outreach). They represent a strong, highly aligned technical fit for our founding team."


### Step 5: Cleanup Temporary Files
We clean up the temporary JSONL file created during Step 2.

In [6]:
if os.path.exists("temp_candidates.jsonl"):
    os.remove("temp_candidates.jsonl")
    print("Temporary file temp_candidates.jsonl cleaned up.")

Temporary file temp_candidates.jsonl cleaned up.
